# What a PDF text extractor really gets

This notebook runs the library blocks of the lesson *Build a small multimodal pipeline*. No GPU, no API key and no model are needed. The outputs saved in the cells come from a real run with the pinned versions.

In [ ]:
!pip install -q reportlab==4.5.1 pypdf==6.16.0 pillow==12.3.0
print('installed')

## 1. Draw a two-page PDF

In [ ]:
import io
from PIL import Image, ImageDraw
from reportlab.lib.utils import ImageReader
from reportlab.pdfgen import canvas

def chart():                                             # the numbers exist only as pixels
    img = Image.new("RGB", (400, 220), "white")
    pen = ImageDraw.Draw(img)
    for i, (month, value) in enumerate([("Jan", 99.95), ("Feb", 99.92), ("Mar", 99.71), ("Apr", 99.97)]):
        x, height = 30 + i * 95, int((value - 99.5) * 340)
        pen.rectangle([x, 190 - height, x + 50, 190], fill="steelblue")
        pen.text((x + 10, 196), month, fill="black")
        pen.text((x + 8, 175 - height), str(value), fill="black")
    return img

def make_pdf():
    out = io.BytesIO()
    page = canvas.Canvas(out, pagesize=(600, 800), invariant=1)
    rows = [("Severity levels", "Rolling back a release"), ("Sev 1: respond in 5 min.", "Run orbit deploy rollback."),
            ("Sev 2: respond in 30 min.", "It takes about 4 minutes.")]
    for n, (left, right) in enumerate(rows):             # written row by row, across both columns
        page.drawString(40, 700 - 30 * n, left)
        page.drawString(320, 700 - 30 * n, right)
    page.showPage()
    table = [("Plan", "Uptime", "Credit"), ("Starter", "none", "none"), ("Business", "99.9", "10")]
    for n, row in enumerate(table):
        for x, cell in zip((40, 180, 300), row):
            page.drawString(x, 740 - 20 * n, cell)
    page.drawImage(ImageReader(chart()), 40, 380, width=400, height=220)
    page.drawString(40, 360, "Figure 1: monthly uptime in percent")
    page.save()
    return out.getvalue()

from pypdf import PdfReader

data = make_pdf()
reader = PdfReader(io.BytesIO(data))
print("file starts with:", data[:5].decode())
print("pages:", len(reader.pages))
print("pictures per page:", [len(p.images) for p in reader.pages])

file starts with: %PDF-
pages: 2
pictures per page: [0, 1]


## 2. Extract the text

In [ ]:
import io
from PIL import Image, ImageDraw
from reportlab.lib.utils import ImageReader
from reportlab.pdfgen import canvas

def chart():                                             # the numbers exist only as pixels
    img = Image.new("RGB", (400, 220), "white")
    pen = ImageDraw.Draw(img)
    for i, (month, value) in enumerate([("Jan", 99.95), ("Feb", 99.92), ("Mar", 99.71), ("Apr", 99.97)]):
        x, height = 30 + i * 95, int((value - 99.5) * 340)
        pen.rectangle([x, 190 - height, x + 50, 190], fill="steelblue")
        pen.text((x + 10, 196), month, fill="black")
        pen.text((x + 8, 175 - height), str(value), fill="black")
    return img

def make_pdf():
    out = io.BytesIO()
    page = canvas.Canvas(out, pagesize=(600, 800), invariant=1)
    rows = [("Severity levels", "Rolling back a release"), ("Sev 1: respond in 5 min.", "Run orbit deploy rollback."),
            ("Sev 2: respond in 30 min.", "It takes about 4 minutes.")]
    for n, (left, right) in enumerate(rows):             # written row by row, across both columns
        page.drawString(40, 700 - 30 * n, left)
        page.drawString(320, 700 - 30 * n, right)
    page.showPage()
    table = [("Plan", "Uptime", "Credit"), ("Starter", "none", "none"), ("Business", "99.9", "10")]
    for n, row in enumerate(table):
        for x, cell in zip((40, 180, 300), row):
            page.drawString(x, 740 - 20 * n, cell)
    page.drawImage(ImageReader(chart()), 40, 380, width=400, height=220)
    page.drawString(40, 360, "Figure 1: monthly uptime in percent")
    page.save()
    return out.getvalue()

from pypdf import PdfReader

first, second = PdfReader(io.BytesIO(make_pdf())).pages
print("--- page 1, default extraction")
print(first.extract_text().strip())
print("--- page 2, default extraction")
text = second.extract_text().strip()
print(text.replace("\n", " / "))
for wanted in ["Business", "Figure 1", "99.71", "Mar"]:
    print(f"{wanted!r} in text of page 2: {wanted in text}")

--- page 1, default extraction
Severity levels
Rolling back a release
Sev 1: respond in 5 min.
Run orbit deploy rollback.
Sev 2: respond in 30 min.
It takes about 4 minutes.
--- page 2, default extraction
Plan / Uptime / Credit / Starter / none / none / Business / 99.9 / 10 / Figure 1: monthly uptime in percent
'Business' in text of page 2: True
'Figure 1' in text of page 2: True
'99.71' in text of page 2: False
'Mar' in text of page 2: False


## 3. Get the picture back

In [ ]:
import io
from PIL import Image, ImageDraw
from reportlab.lib.utils import ImageReader
from reportlab.pdfgen import canvas

def chart():                                             # the numbers exist only as pixels
    img = Image.new("RGB", (400, 220), "white")
    pen = ImageDraw.Draw(img)
    for i, (month, value) in enumerate([("Jan", 99.95), ("Feb", 99.92), ("Mar", 99.71), ("Apr", 99.97)]):
        x, height = 30 + i * 95, int((value - 99.5) * 340)
        pen.rectangle([x, 190 - height, x + 50, 190], fill="steelblue")
        pen.text((x + 10, 196), month, fill="black")
        pen.text((x + 8, 175 - height), str(value), fill="black")
    return img

def make_pdf():
    out = io.BytesIO()
    page = canvas.Canvas(out, pagesize=(600, 800), invariant=1)
    rows = [("Severity levels", "Rolling back a release"), ("Sev 1: respond in 5 min.", "Run orbit deploy rollback."),
            ("Sev 2: respond in 30 min.", "It takes about 4 minutes.")]
    for n, (left, right) in enumerate(rows):             # written row by row, across both columns
        page.drawString(40, 700 - 30 * n, left)
        page.drawString(320, 700 - 30 * n, right)
    page.showPage()
    table = [("Plan", "Uptime", "Credit"), ("Starter", "none", "none"), ("Business", "99.9", "10")]
    for n, row in enumerate(table):
        for x, cell in zip((40, 180, 300), row):
            page.drawString(x, 740 - 20 * n, cell)
    page.drawImage(ImageReader(chart()), 40, 380, width=400, height=220)
    page.drawString(40, 360, "Figure 1: monthly uptime in percent")
    page.save()
    return out.getvalue()

import hashlib
from pypdf import PdfReader

reader = PdfReader(io.BytesIO(make_pdf()))
for number, page in enumerate(reader.pages, 1):
    for found in page.images:
        picture = found.image.convert("RGB")
        print("page", number, "size", picture.size)
        print("same pixels as the chart we drew:", picture.tobytes() == chart().tobytes())
        key = hashlib.sha256(picture.tobytes()).hexdigest()
        print("cache key, first 12 characters:", key[:12])
        print("pointer to keep:", {"page": number, "image": 0, "caption": "Figure 1"})

page 2 size (400, 220)
same pixels as the chart we drew: True
cache key, first 12 characters: 2f3552044503
pointer to keep: {'page': 2, 'image': 0, 'caption': 'Figure 1'}


## Try next

Write the left column completely before the right one and extract again. Use your own files only if they hold no private data.